In [1]:
#meta for myFiftyoneComputerVision 6/12/2025 AML MLP 1 Prep Data - Convert PDFs to Images (maybe create Metadata)
#started from myPDF2Image.ipynb

#env Azure cloud
# default Python 3.8 - AzureML
# $ pip install pdf2image
# $ sudo apt-get install -y poppler-utils (may need to update apt-get)

#history
#6/12/2025 CONVERT PDFs TO IMAGES
#      Make images from PDFs, previously loaded to local directory $config
#      Sample dataset: Mendeley Data, Samples of electronic invoices (from 999 reduced to 250 for speedier POC)

#6/13/2025 CREATE IMAGES METADATA
#      Sample docs most likely don't come with metadata file $config
#      An extra step to create metadata for doc samples
#      Metadata about docs [must/should] accompany images to 51Dataset


In [2]:
import numpy as np
import os
from glob import glob
from pdf2image import convert_from_bytes, convert_from_path
import pandas as pd


In [13]:
#VARS #$config
LOCAL_DIR_SRC = 'DOC-SAMPLE-INVOICES-999_PDFs' 
LOCAL_DIR_TGT = 'DOC-SAMPLE-INVOICES-250_Images'
N = 250 # number of documents to process

METADATA_EXISTS = False #$config

DATA_PATH = 'DOC_DATA'
#metadata out
DATA_SAMPLE_INNOICES_METADATA = DATA_PATH + '/df_sample_invoices_250_metadata.parquet'

In [4]:
#FUNCTIONS

def convert_pdf_to_images(pdf_path, images_path):
    file_name = pdf_path.split('/')[-1]
    images = convert_from_path(pdf_path)
    print(file_name)
    for i in range(len(images)):
        images[i].save(images_path+'/'+file_name[0:-4]+'_PAGE_'+ str(i) +'.jpg', 'JPEG',  )
    return print('SUCCESS:', len(images),' pages saved for: '+file_name)


In [5]:
#!pwd
#out: /mnt/batch/tasks/shared/LS_root/mounts/clusters/compute-ml-doc-vision/code/Users/Anya.Chaliotis

# Source Documents Dataset
with a goal to start with original PDFs and convert PDFs to Images

## 0. Load data
Load PDFs from a local directory

In [6]:
# CHOOSE PDFs FOR PROCESSING
all_documents = glob(LOCAL_DIR_SRC+"/*.pdf")
documents = all_documents[0:N]

len(documents)

250

## 1. Prepare Images
PDFs are one pagers in `sample-invoices` and not one pagers in real world

### 1a. Convert PDFs to Images

In [7]:
# Create a directory to save images
os.makedirs(LOCAL_DIR_TGT, exist_ok=True) #$config

In [8]:
# SPLIT PDF TO IMAGES OF EACH PAGE
for doc in documents:
    convert_pdf_to_images(doc, LOCAL_DIR_TGT)

invoice_0.pdf
SUCCESS: 1  pages saved for: invoice_0.pdf
invoice_1.pdf
SUCCESS: 1  pages saved for: invoice_1.pdf
invoice_10.pdf
SUCCESS: 1  pages saved for: invoice_10.pdf
invoice_100.pdf
SUCCESS: 1  pages saved for: invoice_100.pdf
invoice_101.pdf
SUCCESS: 1  pages saved for: invoice_101.pdf
invoice_102.pdf
SUCCESS: 1  pages saved for: invoice_102.pdf
invoice_103.pdf
SUCCESS: 1  pages saved for: invoice_103.pdf
invoice_104.pdf
SUCCESS: 1  pages saved for: invoice_104.pdf
invoice_105.pdf
SUCCESS: 1  pages saved for: invoice_105.pdf
invoice_106.pdf
SUCCESS: 1  pages saved for: invoice_106.pdf
invoice_107.pdf
SUCCESS: 1  pages saved for: invoice_107.pdf
invoice_108.pdf
SUCCESS: 1  pages saved for: invoice_108.pdf
invoice_109.pdf
SUCCESS: 1  pages saved for: invoice_109.pdf
invoice_11.pdf
SUCCESS: 1  pages saved for: invoice_11.pdf
invoice_110.pdf
SUCCESS: 1  pages saved for: invoice_110.pdf
invoice_111.pdf
SUCCESS: 1  pages saved for: invoice_111.pdf
invoice_112.pdf
SUCCESS: 1  pages sa

### 1b. Persist Images Metadata
if it doesn't exist already

as a dataframe  
`image_file_path`, `image_file_name`

In [9]:
all_images = glob(LOCAL_DIR_TGT+"/*.jpg")
len(all_images)

250

In [10]:
if not METADATA_EXISTS: #$config

    #get image filenames
    all_images_filename = [item[item.index("/")+1:] for item in all_images]
    all_images_filename[:2]

    # Create metadata file
    df_images_metadata = pd.DataFrame(list(zip(all_images,all_images_filename)), columns=['image_file_path','image_file_name'])
    print(df_images_metadata.shape)
    print(df_images_metadata.info())
    print(df_images_metadata.head())

(250, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   image_file_path  250 non-null    object
 1   image_file_name  250 non-null    object
dtypes: object(2)
memory usage: 4.0+ KB
None
                                     image_file_path         image_file_name
0  DOC-SAMPLE-INVOICES-250_Images/invoice_0_PAGE_...    invoice_0_PAGE_0.jpg
1  DOC-SAMPLE-INVOICES-250_Images/invoice_100_PAG...  invoice_100_PAGE_0.jpg
2  DOC-SAMPLE-INVOICES-250_Images/invoice_101_PAG...  invoice_101_PAGE_0.jpg
3  DOC-SAMPLE-INVOICES-250_Images/invoice_102_PAG...  invoice_102_PAGE_0.jpg
4  DOC-SAMPLE-INVOICES-250_Images/invoice_103_PAG...  invoice_103_PAGE_0.jpg


In [14]:
#persist  #$config
if not METADATA_EXISTS:
    df_images_metadata.to_parquet(DATA_SAMPLE_INNOICES_METADATA)

In [12]:
mystop

NameError: name 'mystop' is not defined

## Xtra
### 1b. (Not) Convert Images to Pixels
Next step is to generate image embeddings.  No need to convert images to pixels

In [ ]:
# AI from image to pixels -> 3D array
# from PIL import Image
# import numpy as np

def jpg_to_numpy_pil(image_path):
    img = Image.open(image_path)
    numpy_array = np.array(img)
    return numpy_array

In [ ]:
# CHOOSE PDFs FOR PROCESSING -> chose to do it differently
all_images = glob(LOCAL_DIR_TGT+"/*.jpg")
images = all_images #[0:N]

len(images)

In [ ]:
# CONVERT IMAGES TO PIXELS
l_images = []

for img in images:
    print(img)
    numpy_array = jpg_to_numpy_pil(img)
    l_images.append(numpy_array)

len(l_images)

#preview
l_images[0]

In [ ]:
a_images = np.stack(l_images)
a_images.shape

In [ ]:
a_images_flat = np.reshape(a_images, (a_images.shape[0],a_images.shape[1] * a_images.shape[2] * a_images.shape[3]))
a_images_flat